In [0]:
CREATE OR REPLACE TABLE gold.ic27_tpe_forecast AS
SELECT
  count(*)                                              AS days,
  round(avg(differenceInMinutes), 1)                    AS avg_delay_min,
  percentile_approx(differenceInMinutes, 0.5)           AS median_delay_min,
  max(differenceInMinutes)                              AS max_delay_min,
  round(avg(CASE
          WHEN date_format(from_utc_timestamp(actualTime, 'Europe/Helsinki'), 'HH:mm') <= '16:00'
          THEN 1 ELSE 0 END) * 100, 0)                  AS pct_on_time_by_1600
FROM wojciech_dbx.bronze.trains_arrivals_departures
WHERE stationShortCode = 'TPE'
  AND type = 'ARRIVAL'
  AND cancelled = false
  AND actualTime IS NOT NULL;
-- tylko czwartki: dodaj  AND dayofweek(departureDate) = 5

In [0]:
SELECT
  count(*)                                              AS days,
  round(avg(differenceInMinutes), 1)                    AS avg_delay,
  percentile_approx(differenceInMinutes, 0.5)           AS median_delay,
  max(differenceInMinutes)                              AS max_delay,
  round(avg(CASE
          WHEN date_format(actualTime_local, 'HH:mm') <= '16:00'
          THEN 1 ELSE 0 END) * 100, 0)                  AS pct_on_time_by_1600 /* target time on the event is 16:15, 0:15h transfer, 16:00 MAX arrival */
FROM wojciech_dbx.silver.ic27_thursdays
WHERE cancelled = false

In [0]:
select * from wojciech_dbx.silver.ic27_thursdays

Databricks visualization. Run in Databricks to view.

In [0]:
WITH stats AS (
  SELECT
    count(*)                                        AS days,
    round(avg(differenceInMinutes), 1)              AS avg_delay,
    percentile_approx(differenceInMinutes, 0.5)     AS median_delay,
    max(differenceInMinutes)                        AS max_delay,
    round(avg(CASE
            WHEN date_format(actualTime_local, 'HH:mm:ss') <= '16:00:00'
            THEN 1 ELSE 0 END) * 100, 0)            AS pct_on_time_by_1600  /* target time on the event is 16:15, 0:15h transfer, 16:00 MAX arrival */
  FROM silver.ic27_thursdays
  WHERE cancelled = false
    AND actualTime_local IS NOT NULL
),
last_schedule AS (
  SELECT scheduledTime_local
  FROM silver.ic27_thursdays
  ORDER BY departureDate DESC
  LIMIT 1
)
SELECT
  date_add(to_date(date_trunc('week', current_date())), 10)                          AS forecast_date,
  date_format(l.scheduledTime_local, 'HH:mm')                                        AS scheduled_arrival,
  date_format(timestampadd(MINUTE, s.median_delay, l.scheduledTime_local), 'HH:mm')  AS forecast_arrival,
  s.days,
  s.avg_delay,
  s.median_delay,
  s.max_delay,
  s.pct_on_time_by_1600,
  CASE WHEN s.pct_on_time_by_1600 >= 80
       THEN 'IC27 is fine'
       ELSE 'Take an earlier train'
  END                                                                                AS recommendation
FROM stats s
CROSS JOIN last_schedule l

In [0]:
%python
from pyspark import pipelines as dp

@dp.materialized_view(name='ic27_forecast')
def ic27_forecast():
    return spark.sql("""     
        WITH base AS (
          SELECT
            *,
            row_number() OVER (ORDER BY departureDate DESC) AS rn   /* last thursday's scheduled time (in case of a change in timetable) */
          FROM silver.ic27_thursdays
          WHERE cancelled = false
            AND actualTime_local IS NOT NULL
        ),
        stats AS (
          SELECT
            max(CASE WHEN rn = 1 THEN scheduledTime_local END) AS last_scheduled,
            count(*)                                           AS days,
            round(avg(differenceInMinutes), 1)                 AS avg_delay,
            percentile_approx(differenceInMinutes, 0.5)        AS median_delay, /* median delay will show the most accurate middle figure for the delay, taking huge delays out of the calculation */
            max(differenceInMinutes)                           AS max_delay,
            round(avg(CASE
                    WHEN date_format(actualTime_local, 'HH:mm') <= '16:00'
                    THEN 1 ELSE 0 END) * 100, 0)               AS pct_on_time_by_1600 /* target time on the event is 16:15, 0:15h transfer, 16:00 MAX arrival */
          FROM base
        )
        SELECT
          date_add(to_date(date_trunc('week', current_date())), 10)                AS forecast_date,
          date_format(last_scheduled, 'HH:mm')                                     AS scheduled_arrival,
          date_format(timestampadd(MINUTE, median_delay, last_scheduled), 'HH:mm') AS forecast_arrival,
          days,
          avg_delay,
          median_delay,
          max_delay,
          pct_on_time_by_1600,
          CASE WHEN pct_on_time_by_1600 >= 80 
               THEN 'IC27 is fine'
               ELSE 'Take an earlier train'
          END                                                                      AS recommendation
        FROM stats
    """)   